# Cross-Validation and Preprocessing

**Topic:** Data Preprocessing

In [1]:
import numpy as np
import pandas as pd

import plotly.graph_objects as go

import ipywidgets as widgets
from ipywidgets import RadioButtons, Output, VBox, HTML
from IPython.display import display, clear_output

from sklearn.model_selection import KFold, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline

import seaborn as sns
titanic = sns.load_dataset("titanic")

from tkh_utils import PALETTE, FONT, base_layout

np.random.seed(42)

# Minimal preprocessing so a classifier can run.
df = titanic[["survived", "pclass", "sex", "age", "sibsp", "parch", "fare", "embarked"]].copy()
df["age"]      = df["age"].fillna(df["age"].median())
df["embarked"] = df["embarked"].fillna(df["embarked"].mode()[0])
df["fare"]     = df["fare"].fillna(df["fare"].median())
df = pd.get_dummies(df, columns=["sex", "embarked"], drop_first=False)

X_arr = df.drop(columns="survived").values.astype(float)
y_arr = df["survived"].values

---
## What you'll explore

By the end of this notebook you will be able to:

- **Describe** why a single train-test split may give an unstable estimate of model performance
- **Explain** why preprocessing must happen inside each fold of cross-validation, not on the full dataset first
- **Interpret** the difference between a leaky and a correct cross-validation estimate

> **Tip:** Toggle to "Wrong" first and note the CV score. Then switch to "Correct." The score is slightly lower, but honest. The gap is the amount leakage was inflating the estimate.

---
## How we got here

In `08_data_leakage.ipynb` we learned to fit scalers only on training data. Cross-validation extends that principle: instead of one train-test split, we run K splits and average the scores for a more reliable estimate of performance.

But the same leakage risk from notebook 08 applies inside each fold. This notebook makes that concrete and introduces the tool that solves it.

---
## Why this matters for data science

Cross-validation is the standard way to evaluate a model on limited data, but only if preprocessing happens correctly inside each fold. If you preprocess first and then cross-validate, every fold has already seen every other fold's data through the preprocessing step. The validation fold is no longer truly held out.

---
## Try it yourself

In [ ]:
out = Output()

N_FOLDS = 5
kf = KFold(n_splits=N_FOLDS, shuffle=True, random_state=42)

def run_wrong():
    # Scale ALL data first, then cross-validate: leaky
    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X_arr)
    clf = LogisticRegression(max_iter=1000, random_state=42)
    return cross_val_score(clf, X_scaled, y_arr, cv=kf, scoring="accuracy")

def run_correct():
    # Pipeline: scaling happens inside each fold, no leakage
    pipe = Pipeline([
        ("scaler", StandardScaler()),
        ("clf", LogisticRegression(max_iter=1000, random_state=42)),
    ])
    return cross_val_score(pipe, X_arr, y_arr, cv=kf, scoring="accuracy")

scores_wrong   = run_wrong()
scores_correct = run_correct()

approach_radio = RadioButtons(
    options=["Wrong: preprocess all data, then cross-validate",
             "Correct: Pipeline (preprocessing inside each fold)"],
    value="Wrong: preprocess all data, then cross-validate",
    description="",
    layout=widgets.Layout(width="540px"),
)

def render(change=None):
    approach = approach_radio.value
    is_wrong = "Wrong" in approach
    scores   = scores_wrong if is_wrong else scores_correct
    color    = PALETTE["secondary"] if is_wrong else PALETTE["accent"]

    fold_labels = [f"Fold {i+1}" for i in range(N_FOLDS)]

    fig = go.Figure()
    fig.add_trace(go.Bar(
        x=fold_labels, y=scores,
        marker_color=color,
        text=[f"{s:.3f}" for s in scores],
        textposition="outside",
        showlegend=False,
    ))
    fig.add_hline(
        y=scores.mean(),
        line_dash="dash",
        line_color=color,
        annotation_text=f"Mean: {scores.mean():.4f}",
        annotation_position="top right",
    )

    title = (
        "Leaky CV: Preprocessing Outside Folds"
        if is_wrong else
        "Correct CV: Preprocessing Inside Each Fold (Pipeline)"
    )
    layout = base_layout(title=title, xaxis_title="Fold", yaxis_title="Accuracy")
    layout.update(yaxis=dict(range=[0.60, 0.90]))

    note_html = (
        '<div style="background:#f8d7da;border-left:4px solid #dc3545;'
        'padding:10px 14px;border-radius:4px;margin-top:8px;font-size:13px;">'
        "⚠️ <strong>Leakage present.</strong> The scaler was fit on all 891 rows "
        "before cross-validation ran. Every validation fold has already been seen "
        "by the scaler. The mean accuracy is inflated by approximately "
        f"{scores_wrong.mean() - scores_correct.mean():.4f}.</div>"
        if is_wrong else
        '<div style="background:#d4edda;border-left:4px solid #28a745;'
        'padding:10px 14px;border-radius:4px;margin-top:8px;font-size:13px;">'
        "✔️ <strong>No leakage.</strong> The Pipeline fits the scaler on the "
        "training fold only and transforms the validation fold separately. "
        "This is the honest estimate of model performance.</div>"
    )

    with out:
        clear_output(wait=True)
        display(go.FigureWidget(data=fig.data, layout=layout))
        display(HTML(note_html))
        display(HTML(
            f'<p style="font-size:13px;margin-top:4px;">'
            f"Wrong CV mean: <strong>{scores_wrong.mean():.4f}</strong> | "
            f"Correct CV mean: <strong>{scores_correct.mean():.4f}</strong> | "
            f'Inflation: <strong style="color:{PALETTE["secondary"]};">'
            f"{scores_wrong.mean() - scores_correct.mean():.4f}</strong></p>"
        ))

approach_radio.observe(render, names="value")
display(VBox([approach_radio, out]))
render()

SyntaxError: invalid syntax. Perhaps you forgot a comma? (4100479097.py, line 84)

---
## What's happening?

Cross-validation splits the dataset into K folds and trains K separate models, each using a different fold as the validation set. The final score is the average across all K models. This gives a more stable estimate of performance than a single split.

**The problem:** if you fit a scaler on all the data before cross-validation, every validation fold has been partially seen through the preprocessing step. The validation fold is no longer truly held out.

**The wrong way:**
```python
scaler.fit_transform(X_all)             # touches all rows including validation folds
cross_val_score(clf, X_scaled, y, cv=5)
```

**The right way:** `sklearn.Pipeline` wraps preprocessing and the model together. `cross_val_score` knows to call `fit_transform` on the training fold and `transform` on the validation fold, automatically.

```python
pipe = Pipeline([("scaler", StandardScaler()), ("clf", LogisticRegression())])
cross_val_score(pipe, X, y, cv=5)       # scaler fits inside each fold
```

> **Discussion question:** Why does it matter which fold you fit the scaler on?

| Step | Training fold | Validation fold |
|---|---|---|
| `fit_transform` | ✔️ Applied here | ❌ Never here |
| `transform` | (done by fit_transform) | ✔️ Applied here |
| Model `fit` | ✔️ Training fold only | ❌ Never |
| Model `predict` | (training accuracy) | ✔️ Validation fold only |

---
## Where does this leave us?

For this course, **your preprocessing folder uses a single honest train-test split**. You apply preprocessing manually, fit only on the training set, and save your clean data. That is the approach in the next notebook.

`sklearn.Pipeline`, the tool that makes correct cross-validation automatic, is a model artifact. It bundles your preprocessing and your trained model into one object. It belongs in your **model folder**, alongside the model. You will build it there when you train your first algorithm.

The diagram below shows what happens inside each fold when a Pipeline is used correctly:

In [ ]:
n_samples      = 20
n_folds        = 5
fold_size      = n_samples // n_folds
fold_assignments = np.array([i // fold_size for i in range(n_samples)])

colors = [PALETTE["primary"], PALETTE["accent"], PALETTE["secondary"],
          PALETTE["muted"], "#9775FA"]

fig = go.Figure()

for fold_idx in range(n_folds):
    is_val = (fold_assignments == fold_idx).astype(int)

    for sample_idx in range(n_samples):
        role = "Validate: transform only" if is_val[sample_idx] else "Train: fit + transform"
        c = colors[fold_idx] if is_val[sample_idx] else "#DEE2E6"

        fig.add_trace(go.Scatter(
            x=[sample_idx], y=[fold_idx],
            mode="markers",
            marker=dict(
                symbol="square", size=18, color=c,
                line=dict(color=colors[fold_idx], width=2),
            ),
            showlegend=False,
            hovertemplate=f"Fold {fold_idx+1}, Sample {sample_idx}<br>Role: {role}<extra></extra>",
        ))

layout = base_layout(
    title="5-Fold CV: Colored = Validation Fold (transform only) | Gray = Train Fold (fit + transform)",
    xaxis_title="Sample index",
    yaxis_title="Fold",
)
layout.update(
    yaxis=dict(tickmode="array", tickvals=list(range(n_folds)),
               ticktext=[f"Fold {i+1}" for i in range(n_folds)]),
    height=340,
)
go.FigureWidget(data=fig.data, layout=layout)

---
## Real-world example: A benchmark paper that reported the wrong ranking

A research team compared five classification algorithms on a medical dataset. For all five, they applied MinMax scaling to the full dataset before running cross-validation. Every algorithm's reported accuracy was inflated by the same amount. When the experiment was repeated with Pipelines, the absolute scores dropped and the ranking of two algorithms switched. The paper's conclusion about which algorithm to use in practice was wrong.

---
## Key takeaway

> **Cross-validation only gives you an honest estimate if preprocessing happens inside each fold. In practice, `sklearn.Pipeline` handles this automatically. You will use it in your model folder.**

---
*Next up: 10_saving_your_clean_data.ipynb, completing the manual preprocessing workflow and saving your train/test splits to disk*